# Adult Census Income

Dataset-specific starting notebook for the DataFrameSampler paper experiments.

Claim-specific role: generic mixed-type benchmark, useful for early distributional similarity, downstream utility, and configuration-effort checks.

## Setup

Run the downloader before executing this notebook:

```bash
python experiments/download_datasets.py
```

In [ ]:
from pathlib import Path
import platform
import time

import numpy as np
import pandas as pd

from dataframe_sampler import ConcreteDataFrameSampler

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
DATA = ROOT / "experiments" / "data" / "processed" / "adult.csv"
RESULTS = ROOT / "experiments" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python", platform.python_version())
print("pandas", pd.__version__)
print("dataset", DATA)

In [ ]:
df = pd.read_csv(DATA)
df.shape, df.head()

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(dropna=True),
})
profile

## First DataFrameSampler Run

The full Adult table is moderately large for early exact experiments. This starter notebook uses a fixed subset and the `sklearn` KNN backend so it runs quickly while preserving a reproducible path to the full benchmark.

In [ ]:
work = df.sample(n=2500, random_state=RANDOM_STATE).reset_index(drop=True)

vectorizing_columns_dict = {
    "workclass": ["age", "education_num", "hours_per_week"],
    "education": ["age", "education_num", "hours_per_week"],
    "occupation": ["age", "education_num", "hours_per_week"],
    "marital_status": ["age", "education_num", "hours_per_week"],
    "relationship": ["age", "education_num", "hours_per_week"],
    "native_country": ["age", "education_num", "hours_per_week"],
}

sampler = ConcreteDataFrameSampler(
    n_bins=10,
    n_neighbours=8,
    vectorizing_columns_dict=vectorizing_columns_dict,
    embedding_method="pca",
    knn_backend="sklearn",
    random_state=RANDOM_STATE,
)

start = time.perf_counter()
sampler.fit(work)
fit_seconds = time.perf_counter() - start

start = time.perf_counter()
generated = sampler.sample(n_samples=1000)
sample_seconds = time.perf_counter() - start

fit_seconds, sample_seconds, generated.head()

In [ ]:
def quick_similarity_report(real, synthetic):
    numeric = real.select_dtypes(include="number").columns
    categorical = [c for c in real.columns if c not in numeric]
    rows = []
    for column in numeric:
        rows.append({
            "column": column,
            "kind": "numeric",
            "real_mean": real[column].mean(),
            "synthetic_mean": synthetic[column].mean(),
            "abs_mean_delta": abs(real[column].mean() - synthetic[column].mean()),
            "real_missing": real[column].isna().mean(),
            "synthetic_missing": synthetic[column].isna().mean(),
        })
    for column in categorical:
        real_values = set(real[column].dropna().astype(str))
        synthetic_values = set(synthetic[column].dropna().astype(str))
        rows.append({
            "column": column,
            "kind": "categorical",
            "real_unique": len(real_values),
            "synthetic_unique": len(synthetic_values),
            "category_coverage": len(real_values & synthetic_values) / max(len(real_values), 1),
            "real_missing": real[column].isna().mean(),
            "synthetic_missing": synthetic[column].isna().mean(),
        })
    return pd.DataFrame(rows)

report = quick_similarity_report(work, generated)
report

In [ ]:
generated.to_csv(RESULTS / "adult_generated_start.csv", index=False)
report.to_csv(RESULTS / "adult_similarity_start.csv", index=False)
pd.DataFrame([
    {"dataset": "adult", "rows_used": len(work), "generated_rows": len(generated), "fit_seconds": fit_seconds, "sample_seconds": sample_seconds}
]).to_csv(RESULTS / "adult_runtime_start.csv", index=False)
RESULTS

## Simple Baseline Comparison

Run DataFrameSampler and simple baselines, then summarize them with the TODO point 7 metrics.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

from experiments.compare import run_dataset_comparison

llm_assisted_config = {
    "n_bins": 12,
    "n_neighbours": 8,
    "vectorizing_columns_dict": {
        "workclass": ["age", "education_num", "hours_per_week"],
        "education": ["age", "education_num", "hours_per_week"],
        "marital_status": ["age", "education_num", "hours_per_week", "capital_gain"],
        "occupation": ["age", "education_num", "hours_per_week", "capital_gain"],
        "relationship": ["age", "education_num", "hours_per_week"],
        "race": ["age", "education_num", "hours_per_week"],
        "sex": ["age", "education_num", "hours_per_week"],
        "native_country": ["age", "education_num", "hours_per_week"],
        "income": ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss"],
    },
    "embedding_method": "pca",
    "knn_backend": "sklearn",
    "random_state": RANDOM_STATE,
}

adult_comparison = run_dataset_comparison(
    work,
    dataset_name="adult",
    target_column="income",
    results_dir=RESULTS,
    dataframe_sampler_config={
        "n_bins": 10,
        "n_neighbours": 8,
        "vectorizing_columns_dict": vectorizing_columns_dict,
        "embedding_method": "pca",
        "knn_backend": "sklearn",
        "random_state": RANDOM_STATE,
    },
    llm_assisted_config=llm_assisted_config,
    n_samples=1000,
    random_state=RANDOM_STATE,
)

adult_comparison